# 🚀 Projeto de IA: Previsão de "No-Shows" em Consultas Médicas

**Objetivo:** Construir e avaliar uma rede neural (MLP) para prever a probabilidade de um paciente faltar a uma consulta médica, usando a base de dados pública de Vitória, ES.

**Ferramentas:** Keras/TensorFlow, Pandas, Scikit-learn.
**Dataset:** [Medical Appointment No-Shows no Kaggle](https://www.kaggle.com/datasets/msmstr/medical-appointment-no-shows)

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

# --- CARREGAR OS DADOS ---
# Usamos '..' para "voltar" uma pasta e entrar na pasta 'data'
caminho_dados = '../data/medical_appointments.csv'
df = pd.read_csv(caminho_dados)

print("--- Dados Carregados ---")
df.head()

## 1. Metodologia: Limpeza e Engenharia de Features

Os dados brutos (vistos acima) não estão prontos para a rede neural. Precisamos fazer três transformações principais:

1.  **Limpeza:** Renomear colunas e converter colunas de texto (`Gender`, `NoShow`) para números (0 e 1).
2.  **Engenharia de Features:** As datas (`ScheduledDay`, `AppointmentDay`) são a informação mais valiosa. Vamos usá-las para criar uma *feature* que realmente importa: `WaitingDays` (quantos dias o paciente esperou pela consulta).
3.  **Seleção:** Vamos selecionar apenas as features numéricas e relevantes para o modelo.

In [ ]:

print("--- Iniciando Limpeza e Pré-processamento ---")

# 1. Renomear colunas
df.rename(columns={
    'Hipertension': 'Hypertension',
    'Handcap': 'Handicap',
    'No-show': 'NoShow'
}, inplace=True)

# 2. Converter a variável ALVO (NoShow) para numérico
# SÓ executa o map SE a coluna for do tipo 'object' (texto)
if df['NoShow'].dtype == 'object':
    df['NoShow'] = df['NoShow'].map({'No': 0, 'Yes': 1})
    print("Coluna 'NoShow' mapeada.")

# 3. Converter 'Gender' para numérico
# SÓ executa o map SE a coluna for do tipo 'object' (texto)
if df['Gender'].dtype == 'object':
    df['Gender'] = df['Gender'].map({'M': 0, 'F': 1})
    print("Coluna 'Gender' mapeada.")


# 4. Engenharia de Features de Data (Correção do .normalize)
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay']).dt.tz_localize(None)
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay']).dt.tz_localize(None)
df['ScheduledDay_Date'] = df['ScheduledDay'].dt.normalize()
df['AppointmentDay_Date'] = df['AppointmentDay'].dt.normalize()

df['WaitingDays'] = (df['AppointmentDay_Date'] - df['ScheduledDay_Date']).dt.days
df['WaitingDays'] = df['WaitingDays'].apply(lambda x: 0 if x < 0 else x)

# 5. Criar features de Dia da Semana
df['AppointmentDayOfWeek'] = df['AppointmentDay'].dt.dayofweek

# 6. Seleção Final de Features
features_selecionadas = [
    'Gender',
    'Age',
    'Scholarship',
    'Hypertension',
    'Diabetes',
    'Alcoholism',
    'Handicap',
    'SMS_received',
    'WaitingDays',
    'AppointmentDayOfWeek'
]
target = 'NoShow'

df_limpo = df[features_selecionadas + [target]].copy()

# --- Limpeza Final ---
print(f"\nNulos antes do dropna: {df_limpo.isnull().sum().sum()}")
df_limpo.dropna(inplace=True)

print("\n--- DataFrame Limpo e Pronto para o Modelo ---")
print(f"Formato final: {df_limpo.shape}")
print(df_limpo.head())

## 2. Análise Exploratória: O Desafio Oculto

Com os dados limpos e prontos, o passo mais crítico antes de treinar qualquer modelo é verificar a **distribuição da nossa variável-alvo** (`NoShow`).

Se os dados forem muito desbalanceados (ex: 99% de "Sim" e 1% de "Não"), o modelo pode aprender a "ignorar" a classe minoritária. Vamos verificar.

In [ ]:
print("--- Balanceamento das Classes ---")
print(df_limpo['NoShow'].value_counts(normalize=True))

## 3. Pré-processamento: Preparando os Dados para o Modelo

A análise anterior revelou que temos **~80% de "Compareceu" (0) e ~20% de "Faltou" (1)**. 

Esse desbalanceamento é a principal característica deste problema e vai definir nossa análise final.

Agora, vamos preparar os dados para o Keras:
1.  **Dividir em Treino e Teste:** Usaremos 80% para treino e 20% para teste. Crucialmente, usamos `stratify=y` para garantir que essa proporção de 80/20 se mantenha em ambos os conjuntos.
2.  **Normalização:** Redes neurais são sensíveis a features em escalas diferentes (ex: `Age` de 0-100 vs. `Diabetes` de 0-1). Vamos usar o `StandardScaler` para normalizar todos os dados de entrada.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- 1. Separar Features (X) e Alvo (y) ---
X = df_limpo.drop('NoShow', axis=1)
# y é apenas a coluna alvo que queremos prever
y = df_limpo['NoShow']

print(f"Formato de X (features): {X.shape}")
print(f"Formato de y (alvo): {y.shape}")

# --- 2. Dividir em Treino e Teste ---
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, 
                                                    random_state=42, 
                                                    stratify=y)

# --- 3. Normalização dos Dados (Obrigatório para Redes Neurais) ---
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n--- Dados Prontos ---")
print("Dados de treino normalizados (primeira linha):")
print(X_train_scaled[0])

## 4. Implementação: Construindo a Rede Neural (MLP)

Com os dados normalizados, vamos construir uma arquitetura de **Rede Neural MLP (Multi-layer Perceptron)** simples.

* **Arquitetura:** Usaremos duas camadas ocultas (32 e 16 neurônios) com a função de ativação `ReLU`.
* **Camada de Saída:** Terá 1 neurônio com ativação `sigmoid`, ideal para classificação binária (retorna uma probabilidade entre 0 e 1).
* **Compilador:** Usaremos o otimizador `adam` e a função de perda `binary_crossentropy`, que é a padrão para problemas binários.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# --- 1. Definir a Arquitetura da Rede (MLP) ---
model = Sequential()

model.add(Dense(32, activation='relu', input_shape=(X_train_scaled.shape[1],)))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.summary()
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy']) # Pedimos para ele monitorar a acurácia

# --- 3. Treinar o Modelo ---
print("\n--- Iniciando o Treinamento da Rede Neural ---")

# epochs=10: O modelo vai "ler" todos os dados de treino 10 vezes
# batch_size=32: Ele vai atualizar os pesos a cada 32 amostras
# validation_data: Ele vai testar o modelo nos dados de teste a cada época
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test_scaled, y_test),
    verbose=1 
)

print("--- Treinamento Concluído ---")

## 5. Análise de Resultados: A "Armadilha" da Acurácia

O modelo treinou. Agora, a parte mais importante: a **avaliação crítica**.

Lembre-se do nosso desbalanceamento: um modelo "burro" que sempre previsse "Compareceu" teria 80% de acurácia. Portanto, olhar para a acurácia geral (Accuracy) é uma armadilha.

A métrica que realmente importa para o nosso problema é o **Recall da classe "Faltou (1)"**. Ele responde à pergunta: "De todos os pacientes que *realmente* faltaram, quantos o nosso modelo conseguiu 'pegar'?"

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# --- 1. Fazer as Predições ---
# O modelo 'model.predict' retorna probabilidades (ex: 0.15, 0.82)
y_pred_proba = model.predict(X_test_scaled)

# Convertemos as probabilidades em classes (0 ou 1)
# Se a probabilidade for > 0.5, consideramos "1" (Faltou)
y_pred = (y_pred_proba > 0.5).astype("int32")

# --- 2. Gerar a Matriz de Confusão ---
# A matriz mostra os Verdadeiros Positivos, Falsos Positivos, etc.
print("--- Matriz de Confusão ---")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# --- 3. Gerar o Relatório de Classificação ---
# Este é o relatório mais importante!
# target_names = ['Compareceu (0)', 'Faltou (1)']
print("\n--- Relatório de Classificação ---")
print(classification_report(y_test, y_pred, target_names=['Compareceu (0)', 'Faltou (1)']))

### 5.1. Curvas de Aprendizado (Loss e Accuracy)

O `classification_report` nos diz *o que* o modelo aprendeu (a trapacear). As curvas de Acurácia e Perda (Loss) nos dizem *como* ele aprendeu.

O que buscamos aqui é verificar se houve **overfitting**:
* Se as linhas de Treino e Teste se separarem muito, o modelo "decorou" o treino.
* Se elas andarem juntas (como no nosso caso), o modelo generalizou bem... mesmo que tenha generalizado a "trapaça" do desbalanceamento.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Acurácia do Modelo')
plt.ylabel('Acurácia')
plt.xlabel('Época (Epoch)')
plt.legend(['Treino', 'Teste'], loc='upper left')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Perda (Loss) do Modelo')
plt.ylabel('Perda')
plt.xlabel('Época (Epoch)')
plt.legend(['Treino', 'Teste'], loc='upper right')

plt.tight_layout() 
plt.show()

## 6. Discussão e Conclusão (Aplicação na Engenharia)

### A Descoberta Crítica
O `classification_report` foi a prova final. Nosso modelo atingiu um **Recall de apenas 1%** para a classe "Faltou". Ele identificou corretamente apenas **53 dos 4.464** pacientes que faltaram nos dados de teste.

Isso comprova que uma MLP simples, sem tratamento para o desbalanceamento, **não é viável** para este problema.

### Aplicações na Engenharia
Este modelo, como está, é inútil. No entanto, um modelo *funcional* (provavelmente treinado com técnicas de *resampling* como **SMOTE** ou usando `class_weight`) seria uma ferramenta poderosa para:

* **Engenharia de Produção/Operações:** Otimizar a alocação de equipamentos hospitalares caros, permitindo o "overbooking inteligente" para garantir que a taxa de ociosidade seja mínima.
* **Engenharia de Software:** Ser o "cérebro" de sistemas de gestão de clínicas, enviando lembretes reforçados *apenas* para pacientes com alta probabilidade de falta, otimizando custos e eficiência.